In [10]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, f1_score, confusion_matrix
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, UpSampling1D

# Load the data
train_data_path = r"C:\Users\olufe\projects\Journal\dataset\psa_journal_home_A_unsupervise_train.csv"
test_data_path = r"C:\Users\olufe\projects\Journal\dataset\psa_journal_home_A_sim_test.csv"

train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)

In [11]:
train_data.head()

,M0,M1,M2,M3,M4,M5,M6,M7,M8,M9,...,M1371,M1372,M1373,M1374,M1375,M1376,M1377,M1378,M1379,Label
0,533,597,488,476,387,337,336,333,334,334,...,1055,1033,899,1084,755,436,426,391,407,0
1,371,367,521,508,496,500,490,489,482,491,...,982,376,414,415,416,412,560,571,560,0
2,1033,977,882,855,716,714,763,772,768,767,...,992,962,963,960,983,1108,940,953,932,0
3,466,466,466,462,463,338,338,344,340,368,...,336,337,338,337,339,339,373,336,343,0
4,333,328,336,335,334,333,336,338,336,334,...,1237,593,466,473,461,468,469,470,513,0


In [12]:
test_data.head()

,M0,M1,M2,M3,M4,M5,M6,M7,M8,M9,...,M1371,M1372,M1373,M1374,M1375,M1376,M1377,M1378,M1379,Label
0,285,286,286,286,289,287,429,426,416,416,...,858,1099,937,1036,1065,1020,912,858,788,0
1,680,251,247,249,385,535,527,530,522,424,...,1912,2018,2101,2140,2151,2175,2206,2190,2182,0
2,793,798,829,921,992,1022,918,954,930,993,...,386,379,392,392,384,382,378,380,380,0
3,362,316,288,285,286,289,287,287,288,285,...,833,857,824,817,788,783,808,884,976,1
4,3670,1296,1355,1296,1364,1223,1249,1301,1297,1341,...,492,490,488,479,488,479,472,475,1218,1


In [13]:
# Convert non-numeric values to numeric using LabelEncoder
label_encoder = LabelEncoder()
for column in train_data.columns:
    if train_data[column].dtype == 'object':
        train_data[column] = label_encoder.fit_transform(train_data[column])
        test_data[column] = label_encoder.transform(test_data[column])

# Normalize the data using MinMaxScaler
scaler = MinMaxScaler()
train_data = scaler.fit_transform(train_data)
test_data = scaler.transform(test_data)


In [14]:
print(train_data)

[[0.0583889  0.06992251 0.05027933 ... 0.02916795 0.03162419 0.        ]
 [0.02919445 0.0284736  0.05622635 ... 0.05680074 0.05511207 0.        ]
 [0.14849522 0.13840332 0.12128311 ... 0.11544366 0.11221983 0.        ]
 ...
 [0.0200036  0.0200036  0.02036403 ... 0.02041756 0.01995702 0.        ]
 [0.02036403 0.01964318 0.0200036  ... 0.02026405 0.02026405 0.        ]
 [0.02631105 0.02613083 0.02577041 ... 0.0069082  0.02855388 0.        ]]


In [15]:
print(test_data)

[[0.01369616 0.01387637 0.01387637 ... 0.10085969 0.0901136  0.        ]
 [0.08488016 0.00756893 0.00684808 ... 0.30534234 0.30411422 0.        ]
 [0.10524419 0.10614525 0.11173184 ... 0.02747928 0.02747928 0.        ]
 ...
 [0.04289061 0.03063615 0.03045594 ... 0.29981578 0.29950875 0.        ]
 [0.14020544 0.11083078 0.11930077 ... 0.11252687 0.11436905 1.        ]
 [0.24184538 0.24130474 0.23914219 ... 0.09456555 0.10178078 1.        ]]


In [16]:
#Next, we'll define the 1D-CNN-AE model:
def create_1d_cnn_ae(input_shape):
    input_layer = Input(shape=input_shape)
    encoded = Conv1D(32, 3, activation='relu', padding='same')(input_layer)
    encoded = MaxPooling1D(2, padding='same')(encoded)
    decoded = Conv1D(32, 3, activation='relu', padding='same')(encoded)
    decoded = UpSampling1D(2)(decoded)
    decoded = Conv1D(1, 3, activation='sigmoid', padding='same')(decoded)

    autoencoder = Model(input_layer, decoded)
    autoencoder.compile(optimizer='adam', loss='mean_squared_error')
    return autoencoder


In [17]:
#Now, we'll implement the k-fold cross-validation to estimate model performance 
#and evaluate the model using various performance metrics:

# Define the input shape
input_shape = (train_data.shape[1], 1)

# Perform k-fold cross-validation
k_folds = 5
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

accuracy_scores, precision_scores, recall_scores = [], [], []
tnr_scores, fpr_scores, fnr_scores, f1_scores, auc_scores = [], [], [], [], []

for train_index, val_index in kf.split(train_data):
    x_train, x_val = train_data[train_index], train_data[val_index]
    
    # Reshape the data to fit 1D-CNN-AE input shape
    x_train = x_train.reshape(-1, input_shape[0], 1)
    x_val = x_val.reshape(-1, input_shape[0], 1)
    
    # Create and train the 1D-CNN-AE model
    model = create_1d_cnn_ae(input_shape)
    model.fit(x_train, x_train, epochs=10, batch_size=64, verbose=0)
    
    # Evaluate the model
    val_predictions = model.predict(x_val)
    val_mse = np.mean(np.square(x_val - val_predictions))
    threshold = val_mse * 2.0  # Adjust the threshold based on validation MSE
    
    # Apply the threshold to detect anomalies in the test data
    x_test = test_data.reshape(-1, input_shape[0], 1)
    test_predictions = model.predict(x_test)
    test_mse = np.mean(np.square(x_test - test_predictions))
    
    anomalies = np.where(test_mse > threshold, 1, 0)
    true_labels = np.zeros(len(anomalies))  # Assuming all test samples are normal (0)
    
    # Calculate performance metrics
    accuracy_scores.append(accuracy_score(true_labels, anomalies))
    precision_scores.append(precision_score(true_labels, anomalies))
    recall_scores.append(recall_score(true_labels, anomalies))
    tn, fp, fn, tp = confusion_matrix(true_labels, anomalies).ravel()
    tnr_scores.append(tn / (tn + fp))
    fpr_scores.append(fp / (tn + fp))
    fnr_scores.append(fn / (fn + tp))
    f1_scores.append(f1_score(true_labels, anomalies))
    auc_scores.append(roc_auc_score(true_labels, test_mse))

# Calculate mean and standard deviation of the metrics
mean_accuracy = np.mean(accuracy_scores)
mean_precision = np.mean(precision_scores)
mean_recall = np.mean(recall_scores)
mean_tnr = np.mean(tnr_scores)
mean_fpr = np.mean(fpr_scores)
mean_fnr = np.mean(fnr_scores)
mean_f1 = np.mean(f1_scores)
mean_auc = np.mean(auc_scores)

std_accuracy = np.std(accuracy_scores)
std_precision = np.std(precision_scores)
std_recall = np.std(recall_scores)
std_tnr = np.std(tnr_scores)
std_fpr = np.std(fpr_scores)
std_fnr = np.std(fnr_scores)
std_f1 = np.std(f1_scores)
std_auc = np.std(auc_scores)

# Print the results
print("Mean Accuracy:", mean_accuracy)
print("Mean Precision:", mean_precision)
print("Mean Recall:", mean_recall)
print("Mean TNR:", mean_tnr)
print("Mean FPR:", mean_fpr)
print("Mean FNR:", mean_fnr)
print("Mean F1 Score:", mean_f1)
print("Mean AUC:", mean_auc)

print("Std Accuracy:", std_accuracy)
print("Std Precision:", std_precision)
print("Std Recall:", std_recall)
print("Std TNR:", std_tnr)
print("Std FPR:", std_fpr)
print("Std FNR:", std_fnr)
print("Std F1 Score:", std_f1)
print("Std AUC:", std_auc)


ValueError: in user code:

    File "C:\Users\olufe\anaconda3\lib\site-packages\keras\engine\training.py", line 1284, in train_function  *
        return step_function(self, iterator)
    File "C:\Users\olufe\anaconda3\lib\site-packages\keras\engine\training.py", line 1268, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "C:\Users\olufe\anaconda3\lib\site-packages\keras\engine\training.py", line 1249, in run_step  **
        outputs = model.train_step(data)
    File "C:\Users\olufe\anaconda3\lib\site-packages\keras\engine\training.py", line 1051, in train_step
        loss = self.compute_loss(x, y, y_pred, sample_weight)
    File "C:\Users\olufe\anaconda3\lib\site-packages\keras\engine\training.py", line 1109, in compute_loss
        return self.compiled_loss(
    File "C:\Users\olufe\anaconda3\lib\site-packages\keras\engine\compile_utils.py", line 265, in __call__
        loss_value = loss_obj(y_t, y_p, sample_weight=sw)
    File "